# 01-04 학습 루프로 분류 모델 만들기

런타임 유형을 GPU로 바꾼 뒤 첫 셀부터 차례로 실행하세요.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
print("device:", device)

## 1. 데이터: 원 안쪽과 바깥쪽

In [ ]:
torch.manual_seed(0)
X = torch.rand(400, 2) * 4 - 2  # (400, 2), -2~2 사이의 점
Y = ((X ** 2).sum(dim=1) < 1.5).float()  # (400,), 원 안쪽이면 1

plt.figure(figsize=(4, 4))
plt.scatter(X[:, 0], X[:, 1], c=Y, cmap="coolwarm", edgecolor="k", s=15)
plt.gca().set_aspect("equal")
plt.title("class 1 = inside circle")
plt.show()

X, Y = X.to(device), Y.to(device)

## 2. 모델: 선형 -> ReLU -> 선형

In [ ]:
def init_params(hidden=16):
    torch.manual_seed(0)
    params = {
        "W1": torch.randn(hidden, 2) * 0.5,  # (16, 2)
        "b1": torch.zeros(hidden),  # (16,)
        "W2": torch.randn(1, hidden) * 0.5,  # (1, 16)
        "b2": torch.zeros(1),  # (1,)
    }
    return {k: v.to(device).requires_grad_() for k, v in params.items()}

def model(x, p):
    h = torch.relu(x @ p["W1"].T + p["b1"])  # (N, 16)
    logit = h @ p["W2"].T + p["b2"]  # (N, 1)
    return logit.squeeze(1)  # (N,)

params = init_params()
logit = model(X, params)  # (400,)
print(logit.shape)
print("확률로 바꾼 값:", torch.sigmoid(logit[:5]))

## 3. 손실: 이진 교차 엔트로피

In [ ]:
p = torch.tensor([0.9, 0.6, 0.1])  # 정답(1)에 준 확률
print("-log(p):", (-torch.log(p)).tolist())

loss = F.binary_cross_entropy_with_logits(logit, Y)
print("처음 손실:", loss.item())

## 4. 학습 루프

In [ ]:
def predict_grid(p, f=model, n=200):
    g = torch.linspace(-2, 2, n)
    gx, gy = torch.meshgrid(g, g, indexing="xy")
    pts = torch.stack([gx.flatten(), gy.flatten()], dim=1).to(device)  # (n*n, 2)
    with torch.no_grad():
        prob = torch.sigmoid(f(pts, p))  # (n*n,)
    return gx, gy, prob.reshape(n, n).cpu()

params = init_params()
lr = 0.5
losses, snapshots = [], {}
for step in range(1001):
    logit = model(X, params)  # 1. 순전파
    loss = F.binary_cross_entropy_with_logits(logit, Y)  # 2. 손실
    loss.backward()  # 3. 역전파
    with torch.no_grad():  # 4. 경사 하강법
        for v in params.values():
            v -= lr * v.grad
            v.grad.zero_()  # 5. 기울기 지우기
    losses.append(loss.item())
    if step in (0, 50, 200, 1000):
        snapshots[step] = predict_grid(params)
    if step % 200 == 0:
        acc = ((logit > 0).float() == Y).float().mean()
        print(f"step {step:4d}: loss={loss.item():.4f}, acc={acc.item():.3f}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
Xc, Yc = X.cpu(), Y.cpu()
for ax, (step, (gx, gy, prob)) in zip(axes, snapshots.items()):
    ax.contourf(gx, gy, prob, levels=20, cmap="coolwarm", alpha=0.6, vmin=0, vmax=1)
    ax.contour(gx, gy, prob, levels=[0.5], colors="k")
    ax.scatter(Xc[:, 0], Xc[:, 1], c=Yc, cmap="coolwarm", edgecolor="k", s=8)
    ax.set_aspect("equal")
    ax.set_title(f"step {step}")
axes[4].plot(losses, color="tab:red")
axes[4].set_xlabel("step")
axes[4].set_ylabel("loss")
axes[4].set_title("loss curve")
plt.show()

## 5. 활성화 함수를 빼면

In [ ]:
def linear_model(x, p):
    h = x @ p["W1"].T + p["b1"]  # ReLU 없음
    return (h @ p["W2"].T + p["b2"]).squeeze(1)

params_lin = init_params()
for step in range(1001):
    loss = F.binary_cross_entropy_with_logits(linear_model(X, params_lin), Y)
    loss.backward()
    with torch.no_grad():
        for v in params_lin.values():
            v -= lr * v.grad
            v.grad.zero_()
acc = ((linear_model(X, params_lin) > 0).float() == Y).float().mean()
print(f"ReLU 없는 모델: loss={loss.item():.4f}, acc={acc.item():.3f}")

In [ ]:
gx, gy, prob = predict_grid(params_lin, f=linear_model)
plt.figure(figsize=(4, 4))
plt.contourf(gx, gy, prob, levels=20, cmap="coolwarm", alpha=0.6, vmin=0, vmax=1)
plt.scatter(Xc[:, 0], Xc[:, 1], c=Yc, cmap="coolwarm", edgecolor="k", s=8)
plt.gca().set_aspect("equal")
plt.title("without ReLU")
plt.show()

## 6. 같은 일을 하는 PyTorch 도구

In [ ]:
params = init_params()
optimizer = torch.optim.SGD(params.values(), lr=0.5)
for step in range(1001):
    loss = F.binary_cross_entropy_with_logits(model(X, params), Y)
    optimizer.zero_grad()  # 기울기 지우기
    loss.backward()  # 역전파
    optimizer.step()  # 경사 하강법
print(f"optimizer 사용: loss={loss.item():.4f} (직접 구현: {losses[-1]:.4f})")